# S&P 500 (SPX): Breadth Recovery

Companion to the [Bubble Analytics article](https://www.bubble-analytics.com/blog/SP500-SPX-Breadth-Recovery-2026-10-28), dated 2026-10-28.

Run all cells from this folder after installing `requirements.txt` and JupyterLab.
The included frozen data covers 2016-10-28 to 2026-10-27.
The measured backtest starts on 2017-08-15 after any indicator warmup.
Data files are read locally; no account, API key or network connection is needed to run the backtest.
`TEMPLATE.ipynb` is the existing general strategy template and is separate from this completed example.

The account starts with $100,000. Fees are 0.05% on entry and 0.05% on exit.
Signals are assessed after the close; market entries execute at the next session's open.
Stops and targets, where present, follow the code below. Daily bars cannot establish every intraday price sequence.
Open positions remain marked to market at the end of the sample; closed-trade statistics exclude them.
Spreads, additional slippage, financing and taxes are not modelled.
The benchmark uses the same sample and price convention as the article.

These are historical, in-sample results, not a promise of future performance.

The breadth history uses a fixed snapshot of 503 current constituent securities throughout history. Stocks become eligible once enough price history exists. This introduces survivorship and selection bias: membership is not reconstructed at each historical date. The strategy trades the index, but that selection affects its breadth signals. No historical-membership or out-of-sample comparison is claimed. The benchmark is a price index and excludes dividends.


In [ ]:
import hashlib
from pathlib import Path
input_hashes = {'spx-daily.csv.gz': '3019c4b8eb3f3f7fd48517b8373af45fc001f77ec86e4b90f8316a9cd14b95a5', 'constituent-adjusted-closes.csv.gz': 'cb94adf8b3e3a7dd7046fbbff57f7e71bd5100193a1f3439842fc28eb419f3c4'}
for filename, expected_hash in input_hashes.items():
    assert hashlib.sha256((Path('data')/filename).read_bytes()).hexdigest() == expected_hash, 'Input file changed: ' + filename


## Strategy and backtest


In [ ]:
from IPython.display import display
"""Rebuild the submitted breadth strategy solely from frozen local evidence."""
from pathlib import Path
import json, hashlib, re, warnings
import numpy as np
import pandas as pd
import backtesting
from backtesting import Backtest, Strategy
from pandas.tseries.holiday import (AbstractHolidayCalendar, Holiday, nearest_workday,
    sunday_to_monday, USMartinLutherKingJr, USPresidentsDay, GoodFriday,
    USMemorialDay, USLaborDay, USThanksgivingDay)

ROOT = Path.cwd()
(ROOT/'snapshots').mkdir(exist_ok=True)
(ROOT/'charts').mkdir(exist_ok=True)
OUT = ROOT/'snapshots'
raw = pd.read_csv(ROOT/'data/spx-daily.csv.gz', parse_dates=['Date'], float_precision='round_trip').set_index('Date')
closes = pd.read_csv(ROOT/'data/constituent-adjusted-closes.csv.gz', index_col='Date', parse_dates=True, float_precision='round_trip')
assert raw.index.equals(closes.index)
assert raw.index.is_monotonic_increasing and not raw.index.has_duplicates
assert np.isfinite(raw.to_numpy()).all()
assert (raw[['Open', 'High', 'Low', 'Close']] > 0).all().all()
assert not np.isinf(closes.to_numpy()).any()
assert closes.where(closes.notna(), 1).gt(0).all().all()
symbols = closes.columns.tolist()
def calculate(index_prices, prices):
    ma = prices.rolling(200).mean()
    valid = prices.notna() & ma.notna()
    above = ((prices > ma) & valid).sum(axis=1)
    below = ((prices < ma) & valid).sum(axis=1)
    data = index_prices[['Open','High','Low','Close','Volume']].copy()
    data['Above'],data['Below'] = above,below
    data['Net'] = above/(above+below)
    data = data.dropna().copy()  # Preserve the original strategy's breadth warmup convention.
    hi,lo = data.Net.rolling(14).max(),data.Net.rolling(14).min()
    k = (data.Net-lo)/(hi-lo)*100
    data['Full_K'] = k.rolling(3).mean()
    data['Full_D'] = data.Full_K.rolling(3).mean()
    data['signal'] = np.select([
        (data.Net.shift(1)<.5)&(data.Net>.5),
        (data.Full_K.shift(1)<data.Full_D.shift(1))&(data.Full_K>data.Full_D)], [1,2],default=0)
    assert len(data)==0 or data.signal.iloc[0]==0
    return data

data = calculate(raw,closes)
for endpoint in (210,250,500,1000,2000,len(raw)-1):
    prefix = calculate(raw.iloc[:endpoint],closes.iloc[:endpoint])
    pd.testing.assert_frame_equal(prefix,data.loc[prefix.index])
assert len(data)==2313 and str(data.index[0].date())=='2017-08-15'

class SPXBreadth(Strategy):
    def init(self):
        self.signal = self.I(lambda: self.data.signal)
    def next(self):
        signal = int(self.signal[-1])
        price = float(self.data.Close[-1])
        if signal in (1,2) and (self.position.is_short or not self.position):
            self.position.close()
            self.buy(size=.99, tp=price*(1.02 if signal==1 else 1.03), tag=signal)

bt = Backtest(data, SPXBreadth, cash=100000, margin=1, exclusive_orders=True,
              commission=.0005, trade_on_close=False, finalize_trades=False)
with warnings.catch_warnings(record=True) as captured:
    warnings.simplefilter('always')
    stats = bt.run()
trades = stats['_trades'].copy()
equity = stats['_equity_curve'].copy()
opened = stats['_strategy'].trades
benchmark = (100000*data.Close/data.Close.iloc[0]).rename('BenchmarkEquity')
fees = trades.Size.abs()*(trades.EntryPrice+trades.ExitPrice)*.0005
assert np.allclose(trades.Commission,fees)
assert np.allclose(trades.PnL,trades.Size*(trades.ExitPrice-trades.EntryPrice)-fees)
assert np.allclose(trades.ReturnPct,trades.PnL/(trades.Size*trades.EntryPrice))
trades['SignalDate'] = [data.index[int(i)-1] for i in trades.EntryBar]
trades['SignalClose'] = [data.Close.iloc[int(i)-1] for i in trades.EntryBar]
trades['InitialTarget'] = trades.SignalClose*np.where(trades.Tag==1,1.02,1.03)
trades['CumulativePnL'] = trades.PnL.cumsum()
exposure = np.zeros(len(data),dtype=bool)
closed_exposure = np.zeros(len(data),dtype=bool)
reconstructed = pd.Series(100000.,index=data.index)
realised = reconstructed.copy()
capital = 100000.
for t in trades.itertuples():
    assert data.signal.iloc[t.EntryBar-1]==t.Tag
    assert np.isclose(t.EntryPrice,data.Open.iloc[t.EntryBar])
    assert t.Size==int((capital*.99)//(t.EntryPrice*1.0005))
    assert np.isclose(t.TP,t.InitialTarget) and pd.isna(t.SL)
    assert data.High.iloc[t.ExitBar]>=t.InitialTarget
    assert np.isclose(t.ExitPrice,max(data.Open.iloc[t.ExitBar],t.InitialTarget))
    capital += t.PnL
    exposure[t.EntryBar:t.ExitBar+1] = True
    closed_exposure[t.EntryBar:t.ExitBar+1] = True
    live = (data.index>=t.EntryTime)&(data.index<t.ExitTime)
    reconstructed.loc[live] += t.Size*(data.Close.loc[live]-t.EntryPrice)-t.Size*t.EntryPrice*.0005
    reconstructed.loc[data.index>=t.ExitTime] += t.PnL
    realised.loc[data.index>=t.ExitTime] += t.PnL
for t in opened:
    assert t.size==int((capital*.99)//(t.entry_price*1.0005)) and t.sl is None
    assert np.isclose(t.entry_price,data.Open.iloc[t.entry_bar])
    exposure[t.entry_bar:] = True
    live = data.index>=t.entry_time
    reconstructed.loc[live] += t.size*(data.Close.loc[live]-t.entry_price)-t.size*t.entry_price*.0005
assert np.allclose(reconstructed,equity.Equity)
assert np.isclose(closed_exposure.mean()*100,stats['Exposure Time [%]'])
assert np.isclose((trades.PnL>0).mean()*100,stats['Win Rate [%]'])
assert np.isclose((equity.Equity.iloc[-1]/100000-1)*100,stats['Return [%]'])
assert np.isclose((benchmark.iloc[-1]/100000-1)*100,stats['Buy & Hold Return [%]'])
assert np.isclose((equity.Equity/equity.Equity.cummax()-1).min()*100,stats['Max. Drawdown [%]'])

metrics = {str(k):str(v) if isinstance(v,(pd.Timestamp,pd.Timedelta)) else (float(v) if np.isfinite(v) else None)
           for k,v in stats.items() if not str(k).startswith('_') and isinstance(v,(int,float,np.number,pd.Timestamp,pd.Timedelta))}
metrics.update(symbol='SPX',instrument='S&P 500 price index',initialCash=100000,commissionPerSide=.0005,
    sourceUniverse='Current constituent snapshot applied across history; survivorship bias, not point-in-time membership',
    universeCount=len(symbols),dataFirst=str(raw.index[0].date()),dataLast=str(raw.index[-1].date()),rawRows=len(raw),
    measuredRows=len(data),warmupRows=199,stochasticKFirst=str(data.Full_K.first_valid_index().date()),
    stochasticDFirst=str(data.Full_D.first_valid_index().date()),firstSignal=str(data.index[data.signal.ne(0)][0].date()),
    signalCounts={str(k):int(v) for k,v in data.signal.value_counts().items()},
    entryTypeCounts={str(k):int(v) for k,v in trades.Tag.value_counts().items()},
    eligibleConstituentsMin=int((data.Above+data.Below).min()),eligibleConstituentsMax=int((data.Above+data.Below).max()),
    benchmarkMaxDrawdownPct=float((benchmark/benchmark.cummax()-1).min()*100),benchmarkFinalEquity=float(benchmark.iloc[-1]),
    closedTradePnL=float(trades.PnL.sum()),closedTradeFees=float(fees.sum()),winningTrades=int(trades.PnL.gt(0).sum()),
    losingTrades=int(trades.PnL.lt(0).sum()),realisedMaxDrawdownPct=float((realised/realised.cummax()-1).min()*100),
    openTradesAtEnd=len(opened),openTrades=[dict(entry=str(t.entry_time),entryPrice=float(t.entry_price),size=int(t.size),
        stop=t.sl,target=t.tp,tag=int(t.tag),grossPnL=float(t.size*(data.Close.iloc[-1]-t.entry_price)),
        entryFee=float(t.size*t.entry_price*.0005)) for t in opened],
    pendingOrdersAtEnd=len(stats['_strategy'].orders),backtestingVersion=backtesting.__version__,
    warnings=[str(w.message) for w in captured],
    verification=dict(indicatorPrefixCausality=True,
        fees=True,dailyEquityReconstruction=True,positionSizing=True,entries=True,targets=True,winRate=True,benchmark=True,drawdown=True),
    gapAboveTargetEntryCount=int((trades.EntryPrice>=trades.InitialTarget).sum()),
    sameDayTrades=int((trades.EntryBar==trades.ExitBar).sum()),lastIndexClose=float(data.Close.iloc[-1]),
    originalCAGRCalendar252Pct=metrics['CAGR [%]'],engineClosedTradeExposurePct=metrics['Exposure Time [%]'])
metrics['Exposure Time [%]'] = float(exposure.mean()*100)
metrics['CAGR [%]'] = float(((equity.Equity.iloc[-1]/100000)**(365.2425/(data.index[-1]-data.index[0]).days)-1)*100)
metrics['openTradeNetPnL'] = sum(t['grossPnL']-t['entryFee'] for t in metrics['openTrades'])
metrics['allPaidFees'] = metrics['closedTradeFees'] + sum(t['entryFee'] for t in metrics['openTrades'])
assert np.isclose(100000+metrics['closedTradePnL']+metrics['openTradeNetPnL'],metrics['Equity Final [$]'])
metrics['longestTrade'] = {k:str(v) if isinstance(v,(pd.Timestamp,pd.Timedelta)) else int(v) if isinstance(v,np.integer) else float(v) if isinstance(v,np.floating) and np.isfinite(v) else None if isinstance(v,np.floating) else v
                          for k,v in trades.loc[trades.Duration.idxmax()].to_dict().items()}
metrics['drawdownTroughDate'] = str((equity.Equity/equity.Equity.cummax()-1).idxmin().date())
metrics['randomSeed'] = 20261028
metrics['summedTradeReturnPct'] = float(trades.ReturnPct.sum()*100)
rng = np.random.default_rng(20261028)
curves = np.array([np.r_[0,np.cumsum(rng.permutation(trades.ReturnPct.to_numpy())*100)] for _ in range(100)])
assert np.allclose(curves[:,-1],metrics['summedTradeReturnPct'])
np.save(OUT/'trade-order-curves.npy',curves)
trades.to_csv(OUT/'trades.csv',index=False)
equity.to_csv(OUT/'equity.csv')
data.to_csv(OUT/'indicators.csv')
benchmark.to_csv(OUT/'benchmark.csv')
realised.rename('RealisedBalance').to_csv(OUT/'realised-balance.csv')
(OUT/'results.json').write_text(json.dumps(metrics,indent=2,allow_nan=False)+'\n')
display(pd.Series({k: v for k, v in metrics.items() if not isinstance(v, (dict, list))}, name='Result'))


## Charts and account results


In [ ]:
"""Retain the submitted Plotly panels, colours and fonts; add reconciled comparisons."""
from pathlib import Path
import json
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

ROOT = Path.cwd()
(ROOT/'charts').mkdir(exist_ok=True)
m = json.loads((ROOT/'snapshots/results.json').read_text())
d = pd.read_csv(ROOT/'snapshots/indicators.csv',index_col='Date',parse_dates=True)
t = pd.read_csv(ROOT/'snapshots/trades.csv',parse_dates=['EntryTime','ExitTime'])
e = pd.read_csv(ROOT/'snapshots/equity.csv',index_col=0,parse_dates=True)
b = pd.read_csv(ROOT/'snapshots/benchmark.csv',index_col=0,parse_dates=True).iloc[:,0]
CHARTS = {}
WORDMARK = "<b><span style='color:#00DF9A'>BUBBLE</span><span style='color:white'>ANALYTICS</span></b>"

def brand(fig):
    # Keep branding in the top right header across all chart sizes.
    y = 1 + (fig.layout.margin.t - 34) / (fig.layout.height - fig.layout.margin.t - fig.layout.margin.b)
    x = (fig.layout.width - 24 - fig.layout.margin.l) / (fig.layout.width - fig.layout.margin.l - fig.layout.margin.r)
    fig.add_annotation(text=WORDMARK,x=x,y=y,xref='paper',yref='paper',xanchor='right',yanchor='top',
        showarrow=False,font=dict(size=12,family='Arial, Helvetica, sans-serif'))
    return fig

def themed(fig,title,yaxis):
    fig.update_layout(title=title,template='plotly_dark',autosize=False,width=1100,height=700,
        yaxis_title=yaxis,font=dict(family='Arial, Helvetica, sans-serif',size=14),
        margin=dict(l=95,r=45,t=95,b=90),legend=dict(orientation='h',y=1.03,x=0))
    fig.update_yaxes(gridcolor='#171717');fig.update_xaxes(gridcolor='#171717')
    return brand(fig)

window = d.iloc[2000:2350]
f = make_subplots(rows=3,cols=1,shared_xaxes=True,row_heights=[.50,.25,.25],vertical_spacing=.05)
f.add_trace(go.Candlestick(x=window.index,open=window.Open,high=window.High,low=window.Low,close=window.Close,
    increasing_line_color='rgba(19,156,19,0.8)',decreasing_line_color='rgba(175,07,49,0.8)',name='SPX'),row=1,col=1)
for signal,colour,label,size,symbol in [(1,'white','Breadth recovery',9,'arrow-up'),(2,'gold','Breadth momentum',7,'circle')]:
    f.add_trace(go.Scatter(x=window.index,y=np.where(window.signal==signal,window.Low*.998,np.nan),mode='markers',
        marker=dict(size=size,symbol=symbol,color=colour),name=label),row=1,col=1)
f.add_trace(go.Scatter(x=window.index,y=window.Net,line=dict(color='#00df9a',width=2),opacity=.8,name='Bullish % Index',showlegend=False),row=2,col=1)
f.add_trace(go.Scatter(x=window.index,y=window.Full_K,line=dict(color='red',width=1),name='Bullish % Stochastic',showlegend=False),row=3,col=1)
f.add_trace(go.Scatter(x=window.index,y=window.Full_D,line=dict(color='white',width=1),name='Signal line',showlegend=False),row=3,col=1)
f.add_hline(y=.5,line_width=.5,line_color='grey',row=2,col=1)
for level in (20,80):f.add_hline(y=level,line_width=.5,line_color='grey',row=3,col=1)
f.update_layout(autosize=False,width=1100,height=900,template='plotly_dark',xaxis_rangeslider_visible=False,
    font=dict(family='Arial, Helvetica, sans-serif',size=13),margin=dict(l=75,r=195,t=95,b=85),
    legend=dict(x=1.02,y=1,font=dict(size=11)),
    annotations=[dict(text=text,font=dict(color='white',size=12),xref='paper',yref='paper',x=.03,y=y,showarrow=False)
        for text,y in [('SPX Price Chart',1.02),('Bullish % Index',.5),('Bullish % Stochastic',.24)]])
f.update_xaxes(rangebreaks=[dict(bounds=['sat','mon'])],gridcolor='#171717');f.update_yaxes(gridcolor='#171717')
f.update_yaxes(title_text='Index level',row=1,col=1)
CHARTS['chart-breadth'] = brand(f)

f = go.Figure()
f.add_trace(go.Scatter(x=e.index,y=(e.Equity/100000-1)*100,name='Breadth strategy after fees',line=dict(color='#00df9a',width=2)))
f.add_trace(go.Scatter(x=b.index,y=(b/100000-1)*100,name='SPX price benchmark',line=dict(color='white',width=1.5)))
CHARTS['chart-equity'] = themed(f,'S&P 500 · Account Returns','Cumulative return (%)')
f = go.Figure()
for series,name,colour in [(e.Equity,'Breadth strategy','#00df9a'),(b,'SPX price benchmark','white')]:
    f.add_trace(go.Scatter(x=series.index,y=(series/series.cummax()-1)*100,name=name,line=dict(color=colour,width=1.7)))
CHARTS['chart-drawdown'] = themed(f,'S&P 500 · Daily Closing Drawdowns','Decline from previous peak (%)')
f = go.Figure(go.Scatter(x=[d.index[0],*t.ExitTime,d.index[-1]],y=[0,*t.CumulativePnL,t.CumulativePnL.iloc[-1]],
    mode='lines',name='Closed trade profit',line=dict(color='#00df9a',shape='hv')))
CHARTS['chart-pnl'] = themed(f,'S&P 500 · Closed Trade Profit','Cumulative realised profit ($)')

rows=[('Closed trades',str(int(m['# Trades']))),('Open positions at end',str(m['openTradesAtEnd'])),
      ('Exposure including open position',f"{m['Exposure Time [%]']:.2f}%"),('Closed trade win rate',f"{m['Win Rate [%]']:.2f}%"),
      ('Strategy return after fees',f"{m['Return [%]']:.2f}%"),('SPX price benchmark return',f"{m['Buy & Hold Return [%]']:.2f}%"),
      ('Account maximum drawdown',f"{m['Max. Drawdown [%]']:.2f}%"),('Benchmark maximum drawdown',f"{m['benchmarkMaxDrawdownPct']:.2f}%"),
      ('Final account value',f"${m['Equity Final [$]']:,.0f}"),('Closed trade net profit',f"${m['closedTradePnL']:,.0f}"),
      ('Open position net P&L',f"-${abs(m['openTradeNetPnL']):,.0f}"),('Total fees paid',f"${m['allPaidFees']:,.0f}"),
      ('Best closed trade',f"{m['Best Trade [%]']:.2f}%"),('Worst closed trade',f"{m['Worst Trade [%]']:.2f}%"),
      ('Average trade (geometric)',f"{m['Avg. Trade [%]']:.2f}%"),('Longest closed trade','672 calendar days'),
      ('Sharpe ratio',f"{m['Sharpe Ratio']:.2f}"),('Profit factor','Undefined: no closed losses')]
stripe=['#141A1E' if i%2==0 else '#101519' for i in range(len(rows))]
f = go.Figure(go.Table(columnwidth=[1.7,1.15],header=dict(values=['',''],height=1,fill_color='#161D21',line_color='#161D21',font=dict(size=1,color='#161D21')),
    cells=dict(values=list(zip(*rows)),align=['left','right'],height=34,font=dict(size=12,color='#E6ECEF',family='Arial, Helvetica, sans-serif'),
    fill_color=[stripe,stripe],line_color='rgba(255,255,255,0.08)',line_width=1)))
f.update_layout(width=540,height=105+34*len(rows)+55,paper_bgcolor='#12181B',plot_bgcolor='#161D21',margin=dict(l=24,r=24,t=105,b=55))
f.add_annotation(text='<b>S&P 500 (SPX)<br>Breadth strategy</b>',x=0,y=1.12,xref='paper',yref='paper',xanchor='left',showarrow=False,align='left',font=dict(size=12,color='#E6ECEF',family='Arial, Helvetica, sans-serif'))
brand(f)
f.add_annotation(text='15 August 2017 to 27 October 2026',x=0,y=-.055,xref='paper',yref='paper',showarrow=False,font=dict(size=11,color='#b3c0ca'))
CHARTS['stats-summary'] = f

# Consistent branding with a dedicated header above titles and legends.
wordmark = "<b><span style='color:#00DF9A'>BUBBLE</span><span style='color:white'>ANALYTICS</span></b>"
for figure in CHARTS.values():
    figure.layout.annotations = tuple(a for a in figure.layout.annotations if 'BUBBLE' not in (a.text or ''))
    width, height = figure.layout.width or 1100, figure.layout.height or 700
    figure.update_layout(width=width, height=height, margin_t=max(figure.layout.margin.t or 0, 150))
    if figure.layout.title.text:
        figure.update_layout(title_y=1-65/height, title_yanchor='top')
    inner_height = height - figure.layout.margin.t - (figure.layout.margin.b or 80)
    figure.add_annotation(text=wordmark, x=1, y=1+(figure.layout.margin.t-20)/inner_height,
        xref='paper', yref='paper', xanchor='right', yanchor='top', showarrow=False,
        font=dict(size=12, family='Arial, Helvetica, sans-serif'))


In [ ]:
for figure in CHARTS.values():
    display(figure)
